# Stock Performance Prediction — Preprocessing Pipeline

**Project:** Stock Performance Prediction (RNN)  
**Prerequisite:** `01_exploratory_data_analysis.ipynb`  
**Reference:** `Dataset Cleaning and Alignment Report.md`

This notebook implements the modeling preprocessing pipeline: universe selection → load/validate → clean → features → date alignment → targets → sequences → time split → normalize → export artifacts.

> **After a kernel restart:** run cells sequentially from Section 1, or use **Run All**.

---

## Pipeline overview

```
Raw CSVs → Universe filter → Load & validate → Clean
         → Feature engineering → Merge SPY on Date → Target
         → Sequences → Time split → Normalize (train only) → Artifacts
```

| Stage | Section | Output |
|-------|---------|--------|
| Config | 0–1 | `PIPELINE_CONFIG` |
| Universe | 2 | `eligible_symbols` |
| Load & clean | 3–4 | `cleaned_frames` |
| Features + SPY | 5–6 | `model_df` |
| Sequences | 7–8 | `X_*`, `y_*` arrays |
| Export | 9 | `data/processed/` |

## 0. Design Decisions (v1 defaults)

These defaults follow EDA findings. **Finalize in todo B** before production training.

| Parameter | v1 default | Notes |
|-----------|------------|-------|
| Universe | Stocks only, start with **AAPL** | Expand to multi-symbol after v1 works |
| Min history | 756 trading days (~3 yr) | Supports 60-day lookback + rolling features |
| Target | **Next-day direction** (up=1, down=0) | ~50/50 balance on AAPL in EDA |
| Lookback | 60 trading days | Ablate 30 / 120 later |
| Features | 7 core (see Section 5) | Dedup correlated rolling windows |
| Market factor | SPY log return | Merge on `Date`, never row index |
| Train | ≤ 2017-12-31 | |
| Validation | 2018-01-01 – 2019-12-31 | |
| Test | 2020-01-01 – 2020-04-01 | COVID stress window |

## 1. Setup & Paths

In [1]:
import importlib.util
import json
import subprocess
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

REQUIRED_PACKAGES = ["matplotlib", "numpy", "pandas"]


def ensure_packages(packages):
    missing = [p for p in packages if importlib.util.find_spec(p) is None]
    if missing:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])


ensure_packages(REQUIRED_PACKAGES)
print(f"Python: {sys.executable}")

Python: /Users/franciskanu/Documents/Education/Programing for Data Analytics/.venv/bin/python


In [2]:
def find_project_root() -> Path:
    candidates = [Path.cwd()]
    if Path.cwd().name == "notebooks":
        candidates.append(Path.cwd().parent)
    candidates.extend(Path.cwd().parents)
    for path in candidates:
        if (path / "Stock Market Dataset").exists():
            return path
    return Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "Stock Market Dataset"
STOCKS_DIR = DATA_DIR / "stocks"
ETFS_DIR = DATA_DIR / "etfs"
META_PATH = DATA_DIR / "symbols_valid_meta.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_DIR.exists():
    raise FileNotFoundError(f"Dataset not found at {DATA_DIR}")

print(f"Project root:  {PROJECT_ROOT}")
print(f"Processed dir: {PROCESSED_DIR}")

Project root:  /Users/franciskanu/University/Courses/Semester 2/BIT4.2C - Programming for Data Analytics (SS 26)/Final Project - Stock Performance Prediction
Processed dir: /Users/franciskanu/University/Courses/Semester 2/BIT4.2C - Programming for Data Analytics (SS 26)/Final Project - Stock Performance Prediction/data/processed


## 2. Pipeline Configuration

In [3]:
PIPELINE_CONFIG = {
    # --- Universe ---
    "asset_type": "stock",           # "stock" | "etf"
    "symbols_v1": ["AAPL"],          # start single-symbol; expand in todo B
    "min_trading_days": 756,
    "max_zero_volume_pct": 0.20,       # exclude if >20% zero-volume days
    "financial_status_normal_only": True,
    # --- Features ---
    "feature_cols": [
        "log_return",
        "log_volume",
        "intraday_range",
        "volatility_21",
        "return_ma_21",
        "rsi_14",
        "spy_log_return",
    ],
    "price_col": "Adj Close",
    # --- Target ---
    "target_type": "direction",        # "direction" | "return"
    "target_horizon_days": 1,
    # --- Sequences ---
    "lookback_days": 60,
    "warmup_days": 21,                 # max rolling window used in features
    # --- Splits (inclusive start, inclusive end) ---
    "train_end": "2017-12-31",
    "val_start": "2018-01-01",
    "val_end": "2019-12-31",
    "test_start": "2020-01-01",
    "test_end": "2020-04-01",
    # --- Output ---
    "random_seed": 42,
}

config_path = PROCESSED_DIR / "pipeline_config.json"
with open(config_path, "w") as f:
    json.dump(PIPELINE_CONFIG, f, indent=2)
print(f"Config saved: {config_path}")
display(pd.Series(PIPELINE_CONFIG, name="value"))

Config saved: /Users/franciskanu/University/Courses/Semester 2/BIT4.2C - Programming for Data Analytics (SS 26)/Final Project - Stock Performance Prediction/data/processed/pipeline_config.json


asset_type                                                                  stock
symbols_v1                                                                 [AAPL]
min_trading_days                                                              756
max_zero_volume_pct                                                           0.2
financial_status_normal_only                                                 True
feature_cols                    [log_return, log_volume, intraday_range, volat...
price_col                                                               Adj Close
target_type                                                             direction
target_horizon_days                                                             1
lookback_days                                                                  60
warmup_days                                                                    21
train_end                                                              2017-12-31
val_start       

## 3. Helper Functions

In [4]:
def load_price_data(symbol: str, asset_type: str = "stock") -> pd.DataFrame:
    folder = STOCKS_DIR if asset_type == "stock" else ETFS_DIR
    path = folder / f"{symbol}.csv"
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    df = pd.read_csv(path, parse_dates=["Date"])
    return df.sort_values("Date").reset_index(drop=True)


def validate_ohlcv(df: pd.DataFrame, symbol: str) -> dict:
    """Return quality metrics; raise if hard validation fails."""
    ohlcv = ["Open", "High", "Low", "Close", "Adj Close", "Volume"]
    null_count = int(df[ohlcv].isnull().sum().sum())
    dup_dates = int(df["Date"].duplicated().sum())
    zero_vol = int((df["Volume"] == 0).sum())
    if dup_dates > 0:
        raise ValueError(f"{symbol}: {dup_dates} duplicate dates")
    return {
        "symbol": symbol,
        "rows": len(df),
        "null_count": null_count,
        "zero_volume_days": zero_vol,
        "zero_volume_pct": zero_vol / max(len(df), 1),
        "start_date": df["Date"].min(),
        "end_date": df["Date"].max(),
    }


def clean_ohlcv(df: pd.DataFrame) -> pd.DataFrame:
    """Drop null OHLCV rows; keep sorted by Date."""
    ohlcv = ["Open", "High", "Low", "Close", "Adj Close", "Volume"]
    out = df.dropna(subset=ohlcv).copy()
    return out.sort_values("Date").reset_index(drop=True)


def compute_rsi(series: pd.Series, period: int = 14) -> pd.Series:
    delta = series.diff()
    gain = delta.clip(lower=0).rolling(period).mean()
    loss = (-delta.clip(upper=0)).rolling(period).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))


def engineer_features(df: pd.DataFrame, price_col: str = "Adj Close") -> pd.DataFrame:
    out = df.copy()
    out["log_return"] = np.log(out[price_col] / out[price_col].shift(1))
    out["log_volume"] = np.log1p(out["Volume"])
    out["intraday_range"] = (out["High"] - out["Low"]) / out["Close"]
    out["return_ma_21"] = out["log_return"].rolling(21).mean()
    out["volatility_21"] = out["log_return"].rolling(21).std()
    out["rsi_14"] = compute_rsi(out[price_col], 14)
    return out


def load_spy_returns(price_col: str = "Adj Close") -> pd.DataFrame:
    spy = clean_ohlcv(load_price_data("SPY", asset_type="etf"))
    spy = engineer_features(spy, price_col=price_col)
    return spy[["Date", "log_return"]].rename(columns={"log_return": "spy_log_return"})


def merge_spy_features(df: pd.DataFrame, spy_df: pd.DataFrame) -> pd.DataFrame:
    """Date-aligned join — never stack by row index."""
    return df.merge(spy_df, on="Date", how="inner")


def add_target(df: pd.DataFrame, target_type: str = "direction", horizon: int = 1) -> pd.DataFrame:
    out = df.copy()
    if target_type == "direction":
        out["target"] = (out["log_return"].shift(-horizon) > 0).astype(int)
    elif target_type == "return":
        out["target"] = out["log_return"].shift(-horizon)
    else:
        raise ValueError(f"Unknown target_type: {target_type}")
    return out


print("Helper functions loaded.")

Helper functions loaded.


## 4. Universe Selection

Filter symbols by metadata, history length, nulls, and liquidity.  
For **v1**, we use `symbols_v1` from config; full-universe scan is optional (todo B).

In [5]:
meta = pd.read_csv(META_PATH)
cfg = PIPELINE_CONFIG

if cfg["asset_type"] == "stock":
    meta = meta[meta["ETF"] == "N"].copy()
else:
    meta = meta[meta["ETF"] == "Y"].copy()

if cfg["financial_status_normal_only"]:
    meta = meta[meta["Financial Status"] == "N"].copy()

# v1: explicit symbol list; replace with scan-based filter in todo B
eligible_symbols = [s for s in cfg["symbols_v1"] if (STOCKS_DIR / f"{s}.csv").exists()]
missing = set(cfg["symbols_v1"]) - set(eligible_symbols)
if missing:
    raise FileNotFoundError(f"Configured symbols missing CSV files: {missing}")

eligible_path = PROCESSED_DIR / "eligible_symbols.json"
with open(eligible_path, "w") as f:
    json.dump(eligible_symbols, f, indent=2)

print(f"Eligible symbols ({len(eligible_symbols)}): {eligible_symbols}")
print(f"Saved: {eligible_path}")

Eligible symbols (1): ['AAPL']
Saved: /Users/franciskanu/University/Courses/Semester 2/BIT4.2C - Programming for Data Analytics (SS 26)/Final Project - Stock Performance Prediction/data/processed/eligible_symbols.json


## 5. Load, Validate & Clean (per symbol)

In [6]:
quality_records = []
cleaned_frames = {}

for symbol in eligible_symbols:
    raw = load_price_data(symbol, asset_type=cfg["asset_type"])
    metrics = validate_ohlcv(raw, symbol)
    quality_records.append(metrics)

    if metrics["rows"] < cfg["min_trading_days"]:
        raise ValueError(f"{symbol}: only {metrics['rows']} rows (< {cfg['min_trading_days']})")
    if metrics["zero_volume_pct"] > cfg["max_zero_volume_pct"]:
        raise ValueError(
            f"{symbol}: zero-volume pct {metrics['zero_volume_pct']:.1%} "
            f"> threshold {cfg['max_zero_volume_pct']:.0%}"
        )
    if metrics["null_count"] > 0:
        print(f"WARN {symbol}: {metrics['null_count']} nulls — dropping rows in clean step")

    cleaned = clean_ohlcv(raw)
    cleaned["Symbol"] = symbol
    cleaned_frames[symbol] = cleaned

quality_df = pd.DataFrame(quality_records)
display(quality_df)
print(f"Cleaned {len(cleaned_frames)} symbol(s)")

,symbol,rows,null_count,zero_volume_days,zero_volume_pct,start_date,end_date
0,AAPL,9909,0,0,0.0,1980-12-12,2020-04-01


Cleaned 1 symbol(s)


## 6. Feature Engineering & SPY Merge

Engineer features per symbol, then **inner join on `Date`** with SPY.

In [7]:
spy_features = load_spy_returns(price_col=cfg["price_col"])
featured_frames = {}

for symbol, cleaned in cleaned_frames.items():
    feat = engineer_features(cleaned, price_col=cfg["price_col"])
    feat = merge_spy_features(feat, spy_features)
    feat = add_target(feat, cfg["target_type"], cfg["target_horizon_days"])
    featured_frames[symbol] = feat

# v1 single-symbol convenience frame
model_df = featured_frames[eligible_symbols[0]].copy()
feature_cols = cfg["feature_cols"]

required_cols = feature_cols + ["Date", "Symbol", "target"]
model_df = model_df.dropna(subset=required_cols).reset_index(drop=True)

print(f"Model frame shape: {model_df.shape}")
print(f"Date range: {model_df['Date'].min().date()} → {model_df['Date'].max().date()}")
print(f"Target balance (1=up): {model_df['target'].mean():.3f}")
display(model_df[required_cols].head(3))
display(model_df[required_cols].tail(3))

Model frame shape: (6842, 16)
Date range: 1993-02-01 → 2020-04-01
Target balance (1=up): 0.507


,log_return,log_volume,intraday_range,volatility_21,return_ma_21,rsi_14,spy_log_return,Date,Symbol,target
0,0.028988,17.912159,0.032653,0.027023,0.001181,49.122809,0.007087,1993-02-01,AAPL,0
1,-0.016461,17.635067,0.020747,0.026650,0.001608,37.735972,0.002117,1993-02-02,AAPL,0
2,-0.004158,18.005868,0.041667,0.026437,0.000599,29.166636,0.010514,1993-02-03,AAPL,0


,log_return,log_volume,intraday_range,volatility_21,return_ma_21,rsi_14,spy_log_return,Date,Symbol,target
6839,0.028138,17.553040,0.024018,0.066774,-0.003346,42.224034,0.031960,2020-03-30,AAPL,0
6840,-0.002043,17.712430,0.041252,0.063345,-0.007683,44.345175,-0.015018,2020-03-31,AAPL,0
6841,-0.054052,17.598704,0.039807,0.063943,-0.008720,47.885617,-0.046049,2020-04-01,AAPL,0


## 7. Build Sequences

Create `(n_samples, lookback, n_features)` arrays. Each sample uses only past data (no leakage).

In [8]:
def build_sequences(
    df: pd.DataFrame,
    feature_cols: list,
    target_col: str = "target",
    lookback: int = 60,
    warmup: int = 21,
) -> tuple[np.ndarray, np.ndarray, pd.DatetimeIndex]:
    """Sliding-window sequences from a single-symbol feature dataframe."""
    values = df[feature_cols].to_numpy(dtype=np.float32)
    targets = df[target_col].to_numpy(dtype=np.float32)
    dates = df["Date"]

    start_idx = max(lookback - 1, warmup)
    X_list, y_list, date_list = [], [], []

    for i in range(start_idx, len(df)):
        window = values[i - lookback + 1 : i + 1]
        if window.shape[0] != lookback or np.isnan(window).any():
            continue
        if np.isnan(targets[i]):
            continue
        X_list.append(window)
        y_list.append(targets[i])
        date_list.append(dates.iloc[i])

    return np.array(X_list), np.array(y_list), pd.DatetimeIndex(date_list)


X_all, y_all, seq_dates = build_sequences(
    model_df,
    feature_cols=feature_cols,
    lookback=cfg["lookback_days"],
    warmup=cfg["warmup_days"],
)

print(f"Sequences: X={X_all.shape}, y={y_all.shape}")
print(f"First seq date: {seq_dates.min().date()} | Last: {seq_dates.max().date()}")

Sequences: X=(6783, 60, 7), y=(6783,)
First seq date: 1993-04-27 | Last: 2020-04-01


## 8. Time-Based Train / Val / Test Split

**No shuffling.** Splits follow calendar boundaries from config.

In [9]:
train_end = pd.Timestamp(cfg["train_end"])
val_start = pd.Timestamp(cfg["val_start"])
val_end = pd.Timestamp(cfg["val_end"])
test_start = pd.Timestamp(cfg["test_start"])
test_end = pd.Timestamp(cfg["test_end"])

train_mask = seq_dates <= train_end
val_mask = (seq_dates >= val_start) & (seq_dates <= val_end)
test_mask = (seq_dates >= test_start) & (seq_dates <= test_end)

splits = {
    "train": train_mask,
    "val": val_mask,
    "test": test_mask,
}

X = {"train": X_all[train_mask], "val": X_all[val_mask], "test": X_all[test_mask]}
y = {"train": y_all[train_mask], "val": y_all[val_mask], "test": y_all[test_mask]}

split_summary = pd.DataFrame({
    "split": list(splits),
    "n_samples": [mask.sum() for mask in splits.values()],
    "target_mean": [y_all[mask].mean() for mask in splits.values()],
    "date_start": [seq_dates[mask].min() for mask in splits.values()],
    "date_end": [seq_dates[mask].max() for mask in splits.values()],
})
display(split_summary)

,split,n_samples,target_mean,date_start,date_end
0,train,6217,0.505388,1993-04-27,2017-12-29
1,val,503,0.548708,2018-01-02,2019-12-31
2,test,63,0.428571,2020-01-02,2020-04-01


## 9. Normalize Features (fit on train only)

Per-feature z-score using **training set** statistics only.

In [10]:
def fit_transform_zscore(X_train, X_other_list):
    """Compute mean/std on train; apply to train, val, test."""
    # flatten (n_samples * lookback, n_features) for stable stats
    flat_train = X_train.reshape(-1, X_train.shape[-1])
    mean = flat_train.mean(axis=0)
    std = flat_train.std(axis=0)
    std[std == 0] = 1.0

    def transform(X):
        return (X - mean) / std

    return transform(X_train), [transform(x) for x in X_other_list], mean, std


X_train_norm, [X_val_norm, X_test_norm], feat_mean, feat_std = fit_transform_zscore(
    X["train"], [X["val"], X["test"]]
)

X_norm = {"train": X_train_norm, "val": X_val_norm, "test": X_test_norm}
scaler = {"mean": feat_mean, "std": feat_std, "feature_cols": feature_cols}

print("Normalized shapes:", {k: v.shape for k, v in X_norm.items()})

Normalized shapes: {'train': (6217, 60, 7), 'val': (503, 60, 7), 'test': (63, 60, 7)}


## 10. Export Artifacts

Save arrays and scaler for the modeling notebook (`03_model_training.ipynb`).

In [11]:
artifact_prefix = PROCESSED_DIR / eligible_symbols[0].lower()

np.savez_compressed(
    artifact_prefix.with_name(f"{artifact_prefix.name}_sequences.npz"),
    X_train=X_norm["train"],
    y_train=y["train"],
    X_val=X_norm["val"],
    y_val=y["val"],
    X_test=X_norm["test"],
    y_test=y["test"],
    feature_cols=np.array(feature_cols),
)

with open(PROCESSED_DIR / "scaler.json", "w") as f:
    json.dump(
        {
            "mean": feat_mean.tolist(),
            "std": feat_std.tolist(),
            "feature_cols": feature_cols,
        },
        f,
        indent=2,
    )

model_df.to_csv(PROCESSED_DIR / f"{eligible_symbols[0].lower()}_model_frame.csv", index=False)

print("Exported:")
for p in sorted(PROCESSED_DIR.glob("*")):
    print(f"  {p.name}")

Exported:
  aapl_model_frame.csv
  aapl_sequences.npz
  eligible_symbols.json
  pipeline_config.json
  scaler.json


## 11. Pipeline Validation Checks

In [12]:
checks = {
    "train_samples > 0": X_norm["train"].shape[0] > 0,
    "val_samples > 0": X_norm["val"].shape[0] > 0,
    "test_samples > 0": X_norm["test"].shape[0] > 0,
    "no_nan_in_train_X": not np.isnan(X_norm["train"]).any(),
    "lookback_matches_config": X_norm["train"].shape[1] == cfg["lookback_days"],
    "n_features_matches_config": X_norm["train"].shape[2] == len(feature_cols),
    "train_end_before_val_start": train_end < val_start,
    "val_end_before_test_start": val_end < test_start,
}

check_df = pd.Series(checks, name="passed")
display(check_df)

if not check_df.all():
    failed = check_df[~check_df].index.tolist()
    raise RuntimeError(f"Pipeline validation failed: {failed}")

print("All pipeline checks passed.")

train_samples > 0             True
val_samples > 0               True
test_samples > 0              True
no_nan_in_train_X             True
lookback_matches_config       True
n_features_matches_config     True
train_end_before_val_start    True
val_end_before_test_start     True
Name: passed, dtype: bool

All pipeline checks passed.


---

## 12. Next Steps (todo B & C)

### B — Finalize decisions & extend pipeline
- [ ] Expand universe beyond AAPL (liquid multi-symbol scan)
- [ ] Compare `direction` vs `return` targets
- [ ] Tune lookback (30 / 60 / 120) and feature set
- [ ] Multi-symbol sequence stacking + optional symbol embedding

### C — Strategy document
- [ ] Write `preprocessing-and-modeling-strategy.md` with full rationale

### Modeling (`03_model_training.ipynb`)
- [ ] Load `data/processed/aapl_sequences.npz`
- [ ] Baselines: naive, logistic regression
- [ ] Primary model: GRU / LSTM
- [ ] Compare vs XGBoost on flattened features